# PICK

## 1. Cài đặt môi trường

In [ ]:
import sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("GH_TOKEN")
%cd /kaggle/working
!rm -rf /kaggle/working/reconia
!git clone --depth 1 -b dev -q https://{token}@github.com/tamir39/reconia.git /kaggle/working/reconia
!rm -rf /kaggle/working/reconia/.git
%cd /kaggle/working/reconia
!pip install -q -r requirements.txt

repo_dir = Path("/kaggle/working/reconia")
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

## 2. Chuẩn bị dữ liệu

In [ ]:
from prepare.prepare_sroie import prepare_sroie
from prepare.prepare_cord_v2 import prepare_cord_v2
from prepare.prepare_wildreceipt import prepare_wildreceipt

prepare_sroie()
prepare_cord_v2()
prepare_wildreceipt()

## 3. Huấn luyện mô hình

In [ ]:
from models.pick.train import train_model

# Huấn luyện trên sroie
train_model(dataset_name="sroie", epochs=25, batch_size=4, accum_steps=2, lr=5e-4, patience=5)

In [ ]:
# Huấn luyện trên cord_v2
train_model(dataset_name="cord_v2", epochs=25, batch_size=4, accum_steps=2, lr=5e-4, patience=5)

In [ ]:
# Huấn luyện trên wildreceipt
train_model(dataset_name="wildreceipt", epochs=25, batch_size=4, accum_steps=2, lr=5e-4, patience=5)

## 4. Đánh giá kết quả

In [ ]:
import json
from IPython.display import Markdown, display
from utils.common import DATASETS

rows = [
    "| Dataset | Loss | Precision | Recall | F1 |",
    "| :--- | :---: | :---: | :---: | :---: |",
]
for dataset in DATASETS:
    path = repo_dir / f"results/{dataset}/pick/test_metrics.json"
    if path.exists():
        m = json.loads(path.read_text(encoding="utf-8"))
        rows.append(f"| {dataset} | {m['loss']:.4f} | {m['precision']:.4f} | {m['recall']:.4f} | {m['f1']:.4f} |")

display(Markdown("\n".join(rows)))

## 5. Đóng gói kết quả (Tùy chọn)

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

if Path("checkpoints").exists():
    shutil.make_archive("checkpoints", "zip", "checkpoints")
    display(FileLink("checkpoints.zip"))